In [1]:
import gspread
import pandas
from oauth2client.service_account import ServiceAccountCredentials


from random import random
import time




## Simple example

In [10]:
def authorize_client(key_file, 
                     scope=['https://spreadsheets.google.com/feeds', 
                            'https://www.googleapis.com/auth/drive']
                    ):
    creds = ServiceAccountCredentials.from_json_keyfile_name(key_file, scope)
    client = gspread.authorize(creds)
    print('Ensure files are shared with this email:\n\n  %s' % creds.service_account_email)
    return client

client_secret = 'auth/homebrew-0410c4a011ae.json'
client=authorize_client(key_file=client_secret)
# client.list_spreadsheet_files()

gsheet_file = 'homebrew-log'

sheet = client.open(gsheet_file).sheet1
sheet.get_all_values()



Ensure files are shared with this email:

  log-service@homebrew-247806.iam.gserviceaccount.com


In [54]:
CLEAR_SHEET = True
n_random_rows = 10
header = ['Time', 'Temp', 'Battery']

if CLEAR_SHEET:
    sheet.clear()

if sheet.get_all_values() == []:
    sheet.append_row(header)
    
for row in range(n_random_rows):
    row = [round(random(), 2) for col in header]
    row[0] = time.time()
    sheet.append_row(row)

df = pandas.DataFrame(sheet.get_all_records(), 
                      columns=header)
df

,Time,Temp,Battery
0,1564040616,0.27,0.42
1,1564040616,0.32,0.41
2,1564040616,0.31,0.42
3,1564040616,0.77,0.49
4,1564040617,0.68,0.06
5,1564040617,0.72,0.48
6,1564040617,0.03,0.19
7,1564040617,0.15,0.31
8,1564040618,0.84,0.86
9,1564040618,0.15,0.00


In [66]:
sheet.row_values(1)

DEBUG:requests.packages.urllib3.connectionpool:Resetting dropped connection: sheets.googleapis.com
DEBUG:requests.packages.urllib3.connectionpool:https://sheets.googleapis.com:443 "GET /v4/spreadsheets/1FW5hFbfKyp9FA-C8xlEwXikIok-B6f1JhI1mlpMl0IA/values/Sheet1%21A1%3A1?valueRenderOption=FORMATTED_VALUE HTTP/1.1" 200 None


[u'Time', u'Temp', u'Battery']

## Make a class

In [2]:
import logging
log_level = logging.WARNING # logging.DEBUG
logging.basicConfig(level=log_level)

class gSheetLogger:
    
    def __init__(self, key_file, gsheet_name, sheet_idx=0):
        self.name = gsheet_name
        self.sheet_idx=sheet_idx
        self.service_email = ''
        self.sheet = None
        self.header = None
        self.client = None
        self._authorize_client(key_file)
        self._get_sheet()
    
    def _authorize_client(self,  key_file, 
                     scope=['https://spreadsheets.google.com/feeds', 
                            'https://www.googleapis.com/auth/drive']
                    ):
        try:
            creds = ServiceAccountCredentials.from_json_keyfile_name(key_file, scope)
            self.service_email = str(creds.service_account_email)
            self.client = gspread.authorize(creds)
        except Exception as e:
            logging.error('%s\nFailed to authenticate client...' % e)

    def _get_sheet(self):
        try:
            self.sheet = self.client.open(self.name).get_worksheet(self.sheet_idx)
            logging.debug('Sheet Values:\n%s' % self.sheet.get_all_values())
        except Exception as e:
            logging.error('%s\nEnsure files are shared with this email:\n\n  %s' % (e, self.service_email))
        self._set_header()
            
    def _set_header(self):
        _header = self.sheet.row_values(1) # i guess header starts at row 1
        if _header:
            self.header = _header
            logging.debug('Set header to:\n%s' % self.header)
        else:
            logging.warning('Sheet has no header data!')

    def insert_header(self, header):
        self.sheet.insert_row(header, 1)
        self._set_header()
    
    
    def get_df(self):
        self._set_header()
        return pandas.DataFrame(self.sheet.get_all_records(), 
                      columns=self.header)
    
    def dump_sheet(self, filename, sep='\t'):
        self.get_df().to_csv(filename, sep=sep, header=True)
        
    
    

In [8]:
gs.header

[u'Time', u'Temp', u'Battery']

In [4]:
client_secret = 'auth/homebrew-0410c4a011ae.json'
gsheet_file = 'homebrew-log'
gs = gSheetLogger(key_file=client_secret, gsheet_name=gsheet_file)

In [6]:
# clear sheet, add header
header = ['Time', 'Temp', 'Battery']
gs.sheet.clear()
gs.insert_header(header)
gs.get_df()

,Time,Temp,Battery


In [7]:
# add data
n_random_rows=10
for row in range(n_random_rows):
    row = [round(random(), 2) for col in gs.header]
    row[0] = time.time()
    gs.sheet.append_row(row)
gs.get_df()

,Time,Temp,Battery
0,1564044701,0.16,0.59
1,1564044702,0.55,0.27
2,1564044702,0.84,0.73
3,1564044702,0.43,0.91
4,1564044703,0.98,0.04
5,1564044703,0.80,0.74
6,1564044703,0.48,0.75
7,1564044703,0.02,0.15
8,1564044704,0.67,0.70
9,1564044704,0.08,0.47


In [13]:

datetime.now(timezone('US/Pacific'))

datetime.datetime(2019, 7, 25, 2, 48, 29, 618016, tzinfo=<DstTzInfo 'US/Pacific' PDT-1 day, 17:00:00 DST>)